### inference all spot results and compare

In [1]:
import os

%env LOCAL_PROJECT_DIR=local_project_dir

os.environ["LOCAL_DATA_DIR"] = os.path.join(os.getenv("LOCAL_PROJECT_DIR", os.getcwd()), "data")
os.environ["LOCAL_EXPERIMENT_DIR"] = os.path.join(os.getenv("LOCAL_PROJECT_DIR", os.getcwd()), "classification")
os.environ["LOCAL_SPECS_DIR"] = os.path.join(os.getcwd(), "specs")

%env KEY=dHFmaTA4amVydDR0cG8xOTcwM2FoZW5vMmI6ZjZhOTRmYzUtMjkzOS00NTk2LThkNWEtYzk0NDM1Y2EzZDg3
%env NUM_GPUS=1
%env SPECS_DIR=/workspace/tao-experiments/classification/specs
%env USER_EXPERIMENT_DIR=/workspace/tao-experiments/classification
%env DATA_DOWNLOAD_DIR=/workspace/tao-experiments/data

env: LOCAL_PROJECT_DIR=local_project_dir
env: KEY=dHFmaTA4amVydDR0cG8xOTcwM2FoZW5vMmI6ZjZhOTRmYzUtMjkzOS00NTk2LThkNWEtYzk0NDM1Y2EzZDg3
env: NUM_GPUS=1
env: SPECS_DIR=/workspace/tao-experiments/classification/specs
env: USER_EXPERIMENT_DIR=/workspace/tao-experiments/classification
env: DATA_DOWNLOAD_DIR=/workspace/tao-experiments/data


### copy and convert spot images

In [ ]:
import os
import shutil
from glob import glob

SPOT_INFERENCES = os.path.join(os.getenv('LOCAL_DATA_DIR'), 'spot', 'scanned')
target_dir = os.path.join(os.getenv('LOCAL_DATA_DIR'), 'fulltrain_spot_tests')
if not os.path.exists(target_dir):
  os.makedirs(target_dir)

for x in glob(SPOT_INFERENCES + '/*'):
  shutil.copyfile(x, target_dir + '/' + x.split('/')[-1])

In [4]:
!ls -l $LOCAL_DATA_DIR/fulltrain_spot_tests | wc -l

1112


In [3]:
# lets convert them first
import os
from PIL import Image
from glob import glob
from tqdm import tqdm

LOCAL_DATA_DIR = os.environ.get('LOCAL_DATA_DIR')
images = glob(os.path.join(LOCAL_DATA_DIR, 'fulltrain_spot_tests/*'))

for image in tqdm(images):
  im = Image.open(image)
  im.save(image + '.jpeg', 'JPEG')
  os.remove(image)

100%|██████████| 1112/1112 [00:04<00:00, 251.23it/s]


### inference

In [6]:
# set best epoch
%set_env EPOCH=072

env: EPOCH=072


In [7]:
!tao classification inference -e $SPECS_DIR/fulltrain_classification_retrain_spec.cfg \
                          -m $USER_EXPERIMENT_DIR/fulltrain_output_retrain/weights/resnet_$EPOCH.tlt \
                          -k $KEY -b 32 -d $DATA_DOWNLOAD_DIR/fulltrain_spot_tests \
                          -cm $USER_EXPERIMENT_DIR/fulltrain_output_retrain/classmap.json

2022-02-27 18:23:25,528 [INFO] root: Registry: ['nvcr.io']
2022-02-27 18:23:25,587 [INFO] tlt.components.instance_handler.local_instance: Running command in container: nvcr.io/nvidia/tao/tao-toolkit-tf:v3.21.11-tf1.15.5-py3
Matplotlib created a temporary config/cache directory at /tmp/matplotlib-6ouwx614 because the default path (/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Using TensorFlow backend.
Using TensorFlow backend.

2022-02-27 18:23:36,545 [WARNING] tensorflow: From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:517: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.


2022-02-27 18:23:36,557 [WARNING] tensorflow: From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:4138: The name tf.random_uniform is d